<a href="https://colab.research.google.com/github/ErzatsThesaurus/dsa405-project/blob/main/DSA405_P2_pdcurry_FA26_md.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [38]:
import pandas as pd
import numpy as np
import re


DATA = "https://raw.githubusercontent.com/ErzatsThesaurus/dsa405-project/main/data/raw/"
priceRaw = pd.read_excel(DATA + "APU000072610.xlsx", sheet_name="Monthly")
noaaRaw = pd.read_json(DATA + "noaa_tavg_raw.json", orient="index", convert_axes=False)

In [39]:
# ---- NOAA ----
noaaAudit = noaaRaw.reset_index(names="date")
noaaAudit["date"] = pd.to_datetime(noaaAudit["date"].astype(str), format="%Y%m")

print("NOAA rows, columns:", noaaAudit.shape)

print("\ndtypes:")
print(noaaAudit.dtypes)

print("\nmissing count:")
print(noaaAudit.isna().sum())

print("\nmissing rate (%):")
print(noaaAudit.isna().mean() * 100)

print("\ndistinct values:")
print(noaaAudit.nunique(dropna=False))

print("\nmin:")
print(noaaAudit.min())

print("\nmax:")
print(noaaAudit.max())

print("\nrange:")
print(noaaAudit.max() - noaaAudit.min())

# ---- BLS price ----
print("Price rows, columns:", priceRaw.shape)

print("\ndtypes:")
print(priceRaw.dtypes)

print("\nmissing count:")
print(priceRaw.isna().sum())

print("\nmissing rate (%):")
print(priceRaw.isna().mean() * 100)

print("\ndistinct values:")
print(priceRaw.nunique(dropna=False))

print("\nmin:")
print(priceRaw.min(numeric_only=True))

print("\nmax:")
print(priceRaw.max(numeric_only=True))

print("\nrange:")
print(priceRaw.max(numeric_only=True) - priceRaw.min(numeric_only=True))

NOAA rows, columns: (584, 2)

dtypes:
date     datetime64[ns]
value           float64
dtype: object

missing count:
date     0
value    0
dtype: int64

missing rate (%):
date     0.0
value    0.0
dtype: float64

distinct values:
date     584
value    517
dtype: int64

min:
date     1978-01-01 00:00:00
value                  21.92
dtype: object

max:
date     2026-08-01 00:00:00
value                  76.89
dtype: object

range:
date     17744 days 00:00:00
value                  54.97
dtype: object
Price rows, columns: (573, 2)

dtypes:
observation_date    datetime64[ns]
APU000072610               float64
dtype: object

missing count:
observation_date    0
APU000072610        2
dtype: int64

missing rate (%):
observation_date    0.00000
APU000072610        0.34904
dtype: float64

distinct values:
observation_date    573
APU000072610        118
dtype: int64

min:
APU000072610    0.046
dtype: float64

max:
APU000072610    0.198
dtype: float64

range:
APU000072610    0.152
dtype: float64


In [40]:
sentinels = noaaAudit[noaaAudit["value"] == -99] # per NOAA documentation

print("rows with -99:", len(sentinels))
print(sentinels)

print("rows below 0:", (noaaAudit["value"] < 0).sum())
print("lowest value:", noaaAudit["value"].min())

rows with -99: 0
Empty DataFrame
Columns: [date, value]
Index: []
rows below 0: 0
lowest value: 21.92


In [41]:
temp = noaaRaw.copy()
temp.index = pd.to_datetime(temp.index, format="%Y%m")
temp.index.name = "date"


temp = temp.sort_index().reset_index()


print(len(noaaRaw), "rows in,", len(temp), "rows out")
print("duplicate dates:", temp["date"].duplicated().sum())
print("range:", temp["date"].min().date(), "to", temp["date"].max().date())


expected = pd.date_range(temp["date"].min(), temp["date"].max(), freq="MS")
print("months missing inside the range:", len(expected.difference(temp["date"])))

584 rows in, 584 rows out
duplicate dates: 0
range: 1978-01-01 to 2026-08-01
months missing inside the range: 0


| # | Column | Change made | Rows/cells affected | Why | What is lost |
|---|---|---|---|---|---|
| 1 | `date` | Parsed `YYYYMM` (e.g. `197801`) into dates (`1978-01-01`) | 584 cells | The dates arrived as text, so they could not be manipulated easily| Nothing is removed, but each date now shows a day. Each value is an average for the whole month. Date back to `YYYYMM`. |
| 2 | `date` | Named the unnamed row labels `date` | 1 column label | Needed for clairity and joining | Nothing. Reverse by removing the name. |
| 3 | all | Sorted rows from oldest to newest | 584 rows reordered, 0 removed | Re-ordered to better match time series work that will be done. | The original row order. Reverse by sorting on month, then year. |
| 4 | `date` | Moved the dates from the row labels into a regular column | 584 rows, 1 column added | The join to the price data in P3 matches on a column, and a regular column appears in the data dictionary like any other variable | Nothing. Reverse by setting `date` back as the row labels. |

Working with very clean data set here. They are composed of other raw data from federal agenceies, so there is not much 'cleaning' work to do as there would be with raw temperateure or market data.

| Variable name | Type | Units | Factor levels | Valid range | Missingness | Notes |
|---|---|---|---|---|---|---|
| `date` | date | calendar month | not categorical | Any month from 1978-01 through the most recent month NOAA has published | 0 missing among 584 rows. September through December 2026 are absent as whole rows, not as blank values, because NOAA had not published them when the data was pulled. | The source records year and month only. The day shown (always the 1st) was added when the text key was parsed and carries no meaning. Each row covers the whole month. |
| `value` | float | degrees Fahrenheit | not categorical | Roughly 15 to 85. A monthly average for the whole contiguous U.S. outside that band would signal an error. | 0 missing. NOAA marks missing values with -99; none are present. | Average temperature across the contiguous 48 states (no Alaska or Hawaii), not a reading from any one place. The units are not stated in the saved file; they come from NOAA's documentation. The most recent months are preliminary and NOAA may revise them. Observed range: 21.92 to 76.89. |

| | Count |
|---|---|
| Rows in raw file | 584 |
| Rows removed as exact duplicates | 0 |
| Rows removed as near-duplicates | 0 |
| Rows removed for other reasons | 0 |
| **Rows in cleaned file** | **584** |
| | |
| Columns in raw file | 1 (`value`; the dates were row labels, not a column) |
| Columns dropped | 0 |
| Columns added | 1 (`date`, moved out of the row labels; log row 4) |
| **Columns in cleaned file** | **2** |


Once again, I am working with very clean data from government agenceys. I am several steps removed from actual 'raw' data.'

**Provenance Brief**

This project uses data sources from two U.S. government agencies. The noaaRaw data set is average temperateure data from the contiguous united states from each month, going back to the 70's. The second is the average price households in the U.S. paid for one kilowatt-hour of electricity every month, also going back to the 70's.  Together they can show what kind of corralation exists between the weather and the price of electricity.

AI USE NOTE:

I used Claude for codeing synatx and for generating markdown tables.